# 06 — Check whether all project tables are empty
Inspect every configured Bronze, Silver, Gold, datamart, and operations table. Missing tables count as empty. The result is published as the Lakeflow Jobs task value `tables_empty`.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / "src" for root in (Path.cwd(), *Path.cwd().parents) if (root / "src").is_dir()), None)
if source_root is None:
    raise FileNotFoundError("Open this notebook from the FinOps Cloud Data Platform Git Folder.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = "dev"
dbutils.widgets.dropdown("environment", ENVIRONMENT, ["dev", "prod"])
ENVIRONMENT = dbutils.widgets.get("environment")

In [ ]:
from finops_cloud.config import load_config
from finops_cloud.maintenance import all_tables_empty, inspect_project_tables
from finops_cloud.runtime import get_spark

config = load_config(ENVIRONMENT)
spark_session = get_spark(config.profile)
states = inspect_project_tables(spark_session, config)
tables_empty = all_tables_empty(states)
non_empty_tables = [state["table_name"] for state in states if not state["is_empty"]]
dbutils.jobs.taskValues.set(key="tables_empty", value=tables_empty)
dbutils.jobs.taskValues.set(key="non_empty_count", value=len(non_empty_tables))
print(f"tables_empty={str(tables_empty).lower()}; non_empty_count={len(non_empty_tables)}")
display(spark_session.createDataFrame(states).orderBy("layer", "table_name"))